# Activity 8.02 - Visualizing City Density by the First Letter



## 1. Import dependencies

In [1]:
import pandas as pd
import geoplotlib
from geoplotlib.layers import BaseLayer
from geoplotlib.core import BatchPainter
from geoplotlib.utils import DataAccessObject, BoundingBox
import pyglet.window.key as key

## 2. Load world_cities_pop.csv

In [2]:
dataset = pd.read_csv('world_cities_pop.csv',
                    dtype={'Region': str})
dataset.head()

,Country,City,AccentCity,Region,Population,Latitude,Longitude
0,ad,aixas,Aixàs,06,NaN,42.483333,1.466667
1,ad,aixirivali,Aixirivali,06,NaN,42.466667,1.500000
2,ad,aixirivall,Aixirivall,06,NaN,42.466667,1.500000
3,ad,aixirvall,Aixirvall,06,NaN,42.466667,1.500000
4,ad,aixovall,Aixovall,06,NaN,42.466667,1.483333


## 3. Rename Latitude/Longitude to lat/lon

In [3]:
dataset = dataset.rename(columns={'Latitude': 'lat', 'Longitude': 'lon'})
dataset.head()

,Country,City,AccentCity,Region,Population,lat,lon
0,ad,aixas,Aixàs,06,NaN,42.483333,1.466667
1,ad,aixirivali,Aixirivali,06,NaN,42.466667,1.500000
2,ad,aixirivall,Aixirivall,06,NaN,42.466667,1.500000
3,ad,aixirvall,Aixirvall,06,NaN,42.466667,1.500000
4,ad,aixovall,Aixovall,06,NaN,42.466667,1.483333


## 4. Filter to European cities

In [4]:
europe_country_codes = ['al','ad','at','by','be','ba','bg','hr','cz','dk','ee',
                        'fi','fr','de','gr','hu','is','ie','it','lv','li','lt',
                        'lu','mk','mt','md','mc','nl','no','pl','pt','ro','ru',
                        'sm','rs','sk','si','es','se','ch','ua','gb','va']

europe = dataset[dataset['Country'].isin(europe_country_codes)]
print('All cities:', len(dataset))
print('European cities:', len(europe))

All cities: 3173958
European cities: 890937


## 5. Filter cities starting with Z

In [5]:
europe_z = europe[europe['City'].str.lower().str.startswith('z', na=False)]
print('Cities starting with Z:', len(europe_z))
europe_z.head()

Cities starting with Z: 20574


,Country,City,AccentCity,Region,Population,lat,lon
104206,al,zaane,ZÄane,44,NaN,40.932778,19.783056
104207,al,zabarzani,Zabarzani,40,NaN,40.427778,20.269167
104208,al,zabarzan,Zabarzan,40,NaN,40.427778,20.269167
104209,al,zaberzane,Zabërzanë,40,NaN,40.427778,20.269167
104210,al,zaberzan i siperm,Zaberzan i Sipërm,40,NaN,40.427778,20.269167


## 6. Dot density plot with tooltip

In [6]:
europe_data = DataAccessObject(europe_z)
geoplotlib.dot(europe_data, f_tooltip=lambda d: d['Country'] + ' - ' + d['AccentCity'],
               point_size=2)
geoplotlib.set_bbox(BoundingBox(north=71, west=-25, south=35, east=45))
geoplotlib.inline()

## 7. Voronoi of Z cities

In [7]:
geoplotlib.voronoi(europe_z, cmap='Reds_r', max_area=1e5, alpha=128)
geoplotlib.set_bbox(BoundingBox(north=71, west=-25, south=35, east=45))
geoplotlib.inline()

## 8. Interactive FilterLayer

In [8]:
europe_bbox = BoundingBox(north=71, west=-25, south=35, east=45)


class FilterLayer(BaseLayer):
    def __init__(self, data, bbox=BoundingBox.WORLD):
        self.data = data
        self.start_letter = 0
        self.start_letters = [chr(c) for c in range(ord('A'), ord('Z') + 1)]
        self.view = bbox

    def invalidate(self, proj):
        self.painter = BatchPainter()
        letter = self.start_letters[self.start_letter]
        filtered = self.data[self.data['City'].str.lower().str.startswith(letter.lower(), na=False)]
        x, y = proj.lonlat_to_screen(filtered['lon'], filtered['lat'])
        self.painter.points(x, y, 2)

    def draw(self, proj, mouse_x, mouse_y, ui_manager):
        self.painter.batch_draw()
        ui_manager.info('Displaying cities starting with {} (N=next, P=prev)'.format(
            self.start_letters[self.start_letter]))

    def on_key_release(self, key_code, modifiers):
        if key_code in (key.RIGHT, key.BRACKETRIGHT, key.N):  # next letter
            self.start_letter = (self.start_letter + 1) % 26
            return True
        elif key_code in (key.LEFT, key.BRACKETLEFT, key.P):  # previous letter
            self.start_letter = (self.start_letter - 1) % 26
            return True
        return False

    def bbox(self):
        return self.view

## 9. Run the interactive layer

In [10]:
europe_data = DataAccessObject(europe)
geoplotlib.add_layer(FilterLayer(europe, bbox=europe_bbox))
geoplotlib.show()